# 03 – Sensitivity (E4) and layer ablation (E8)

Which assumptions matter for the exposed population: threshold error, satellite frequency, stage error, population
noise, DEM error (E4), and the dike layer and satellite calibration themselves (E8). Totals are people in the 54
communes. Monte Carlo uses seed 42; its results depend on the random stream, so they agree with the paper
within sampling variation rather than digit for digit.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
pd.options.display.float_format = "{:,.3f}".format

from analysis.flood_model import FloodModel
from analysis.sensitivity import Sensitivity
fm = FloodModel(ROOT / "data/derived/grids_public.npz", ROOT / "data/mrc_derived/calib.json")
s = Sensitivity(fm)

## E4 at Alarm level I (3.50 m)

In [2]:
s.e4(3.5, n_mc=100)

,level,source,people_lo,people_hi,shift_%,rho_median,rho_p5
0,3.500,baseline,"165,285.891","165,285.891",0.000,1.000,1.000
1,3.500,"threshold error sigma=0.1 m, 2 km correlation","153,703.016","171,929.656",-2.262,0.983,0.974
2,3.500,"threshold error sigma=0.25 m, 2 km correlation","152,846.719","182,466.828",1.704,0.946,0.917
3,3.500,satellite frequency x0.8,"122,652.406","122,652.406",-25.794,0.975,0.975
4,3.500,satellite frequency x1.2,"200,802.219","200,802.219",21.488,0.982,0.982
5,3.500,stage -0.157 m,"136,920.312","136,920.312",-17.162,0.989,0.989
6,3.500,stage +0.157 m,"200,802.219","200,802.219",21.488,0.982,0.982
7,3.500,population noise sigma_log=0.3,"164,236.515","166,383.143",0.041,0.998,0.996


## E4 at 4.70 m (the map is saturated between 4.63 and 5.06 m)

In [3]:
s.e4(4.7, n_mc=100)

,level,source,people_lo,people_hi,shift_%,rho_median,rho_p5
0,4.700,baseline,"607,942.000","607,942.000",0.000,1.000,1.000
1,4.700,"threshold error sigma=0.1 m, 2 km correlation","567,349.812","592,389.625",-4.688,0.995,0.991
2,4.700,"threshold error sigma=0.25 m, 2 km correlation","535,536.562","572,682.750",-8.801,0.988,0.980
3,4.700,satellite frequency x0.8,"607,942.000","607,942.000",0.000,1.000,1.000
4,4.700,satellite frequency x1.2,"607,942.000","607,942.000",0.000,1.000,1.000
5,4.700,stage -0.157 m,"495,697.312","495,697.312",-18.463,0.991,0.991
6,4.700,stage +0.157 m,"607,942.000","607,942.000",0.000,1.000,1.000
7,4.700,population noise sigma_log=0.3,"605,814.788","610,003.412",-0.002,0.999,0.999


## E4 – DEM error and depth

People in cells deeper than 1 m depend on the DEM (error σ = 2 m, correlation 1 km); this is why depth is not used as a result.

In [4]:
s.e4_depth(4.7, 50)

{'level': 4.7,
 'baseline_people_gt_1m': np.float32(178818.4),
 'p5': np.float32(204033.47),
 'median': np.float32(243512.16),
 'p95': np.float32(289737.66)}

## E8 – ablation of the spatial layers (paper Table `e8`)

In [5]:
s.e8()

,level,configuration,people,change_%,land_km2,IoU_vs_full,spearman
0,3.500,"Full: satellite thresholds, with dikes","165,285.891",NaN,"1,990.621",1.000,1.000
1,3.500,No dike layer,"258,580.906",56.445,"2,350.185",0.847,0.924
2,3.500,"DEM only, tilted surface, connected","215,387.109",30.312,"2,230.536",0.658,0.762
3,3.500,"DEM only, flat surface, connected","445,441.312",169.497,"2,404.337",0.616,0.668
4,4.700,"Full: satellite thresholds, with dikes","607,942.000",NaN,"2,822.165",1.000,1.000
5,4.700,No dike layer,"924,712.250",52.105,"3,470.994",0.813,0.937
6,4.700,"DEM only, tilted surface, connected","797,784.125",31.227,"3,310.485",0.765,0.885
7,4.700,"DEM only, flat surface, connected","1,027,917.125",69.081,"3,167.342",0.711,0.827


**Known difference.** For *DEM only, flat surface* at 4.70 m the paper reports 1,058,948 people (Spearman 0.834); this
code gives 1,027,917 (0.827). The other seven rows of the table are reproduced exactly. The flat-surface rule used for
that single row in the original script should be checked.